In [10]:
import sys
sys.path.insert(0, '..')
from dependencies import *

envelopes_log = eelbrain.load.unpickle(PROCESSED_PREDICTOR_DIR / f'~processed_envelopes-log.pickle')
durations = get_durations(envelopes_log)
models = get_models()

In [ ]:
# PREDICT ENVELOPES FROM THE EEG DATA

subject = 'S05'
model = 'envelope_log_8band'
trf_decoder = eelbrain.load.unpickle(TRF_DIR / f'{subject}' / f'{subject} decoder-{model}.pickle')
eeg = eelbrain.load.unpickle(STIMULUS_DIR / f'{subject}concatenated_eeg.pickle')
subject_model_predictor = eelbrain.load.unpickle(PREDICTOR_DIR / f'~concatenated_predictors.pickle')
envelope = subject_model_predictor[subject][model]

#print(trf_s05.h_scaled)
#print(envelopes_log)

predicted_envelope = eelbrain.convolve(trf_decoder.h_scaled, eeg)
# Convert NDVars to numpy arrays for plotting
envelope_data = envelope.x             # shape: (frequency, time)
time = envelope.time
#print(predicted_envelope)
#print(envelope_data)

DimensionMismatchError: Incompatible dimensions
h: <Sensor n=61, name=None>
x: <Sensor n=55, name=None>

In [6]:
# Extract numpy arrays
env = envelope.x                 # shape: (bands, time)
pred = predicted_envelope.x

n_bands = env.shape[0]

# Compute Pearson r per band
r_values = np.array([
    np.corrcoef(env[i], pred[i])[0, 1]
    for i in range(n_bands)
])

# Compute r²
r_squared_values = r_values ** 2

# Print nicely
for i in range(n_bands):
    print(f"Band {i}: r = {r_values[i]:.4f}, r² = {r_squared_values[i]:.4f}")


/Users/drmanistein/miniforge3/envs/eelbrain/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/drmanistein/miniforge3/envs/eelbrain/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/drmanistein/miniforge3/envs/eelbrain/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


Band 0: r = nan, r² = nan
Band 1: r = nan, r² = nan
Band 2: r = nan, r² = nan
Band 3: r = nan, r² = nan
Band 4: r = nan, r² = nan
Band 5: r = nan, r² = nan
Band 6: r = nan, r² = nan
Band 7: r = nan, r² = nan
Band 8: r = nan, r² = nan
Band 9: r = nan, r² = nan
Band 10: r = nan, r² = nan
Band 11: r = nan, r² = nan
Band 12: r = nan, r² = nan
Band 13: r = nan, r² = nan
Band 14: r = nan, r² = nan
Band 15: r = nan, r² = nan
Band 16: r = nan, r² = nan
Band 17: r = nan, r² = nan
Band 18: r = nan, r² = nan
Band 19: r = nan, r² = nan
Band 20: r = nan, r² = nan
Band 21: r = nan, r² = nan
Band 22: r = nan, r² = nan
Band 23: r = nan, r² = nan
Band 24: r = nan, r² = nan
Band 25: r = nan, r² = nan
Band 26: r = nan, r² = nan
Band 27: r = nan, r² = nan
Band 28: r = nan, r² = nan
Band 29: r = nan, r² = nan
Band 30: r = nan, r² = nan
Band 31: r = nan, r² = nan
Band 32: r = nan, r² = nan
Band 33: r = nan, r² = nan
Band 34: r = nan, r² = nan
Band 35: r = nan, r² = nan
Band 36: r = nan, r² = nan
Band 37: r 

In [7]:
import matplotlib.pyplot as plt
import numpy as np

# Extract arrays
env = envelope.x
pred = predicted_envelope.x
time = envelope.time.times   # ← FIXED (extract numeric time values)

n_bands = env.shape[0]

# --------------------------------------------
# Restrict to first 50 seconds
# --------------------------------------------
t_max = 50
time_mask = time <= t_max

time_50 = time[time_mask]
env_50 = env[:, time_mask]
pred_50 = pred[:, time_mask]

# Scale predicted envelope for visualization ONLY
pred_scaled_50 = pred_50 * (env_50.std(axis=1) / pred_50.std(axis=1))[:, None]
#pred_scaled_50 = pred_50

# --------------------------------------------
# Plot each band in its own subplot
# --------------------------------------------
fig, axes = plt.subplots(n_bands, 1, figsize=(12, 2.5*n_bands), sharex=True)

for i in range(n_bands):
    ax = axes[i]
    ax.plot(time_50, env_50[i], label='Actual')
    ax.plot(time_50, pred_scaled_50[i], label='Predicted (scaled)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'Frequency Band {i}')
    ax.legend(loc='upper right')

axes[-1].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()


# --------------------------------------------
# Bar plot of r² (computed on full data)
# --------------------------------------------
plt.figure(figsize=(8, 4))
plt.bar(range(n_bands), r_squared_values)
plt.xlabel('Frequency Band')
plt.ylabel('Proportion Explained (r²)')
plt.title(f'Decoder Performance for {subject} - {model}')
plt.xticks(range(n_bands))
plt.ylim(0, max(r_squared_values)*1.1)
plt.tight_layout()
plt.show()



IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed